# Fifa Prediction model

Each training example will be a (22,9) matrix. Each row is a player, the first 11 rows are players from the home team while the last 11 rows are players from the away team. For each player, the following details are used:

1. age
2. is forward
3. is mid fielder
4. is defense
5. is goalkeeper
6. number tournaments
7. appearances
8. goals
9. average time per team

This matrix will be passed to the first layer of the neural network that should predict the 'quality' of each player in the team. There will be a ReLU activation for this network. This will be done by doing:

$$
p = xw^T + b \\
ap = max(0, p)
$$

The output of this will be a (22, 1) matrix. This will then be passed to the team wins prediction portion of the network. This has 1 layer with 121 units with softmax activation as shown below:

$$
L = 1 \\
n^{[1]} = 2 \\
z = apw + b \\
y = softmax(z)
$$

The goal is to get a model that surpasses the following benchmarks:
- 75.42% accuracy on predicting a win
- 18% accuracy for predicting exact scores

In [112]:
# Imports
import numpy as np
import numpy.typing as npt
from sklearn.model_selection import train_test_split
import math

In [113]:
def f_x(
    players: npt.NDArray, 
    w_p: npt.NDArray, 
    b_p: float, 
    w_t: npt.NDArray, 
    b_t: float
):
    """ 
    Function that will do a forward pass for a single training example in the example above.
    
    Args:
        players (ndarray) - a (22,9) numpy array with 9 features for 22 players from the home and away team
        w_p (ndarray) - a (1,9) numpy array with weights for player predictions
        b_p (scalar) - bias for player quality prediction
        w_t (ndarray) - a (121, 22) numpy array with the weights for the team score prediction
        b_t (scalar) - bias for the team score prediction
        
    Returns:
        scores (ndarray) - a (121,1) numpy array with probability of each of the score distributions
        z_t (ndarray) - cached z_t value
        a_p (ndarray) - cached a_t value,
        z_p (ndarray) - cached z_p value
    """
    z_p = np.matmul(players, w_p.T) + b_p
    a_p = np.maximum(0, z_p)
    z_t = np.matmul(w_t, a_p) + b_t
    shifted_logits = z_t - np.max(z_t, axis=0, keepdims=True)
    e_zi = np.exp(shifted_logits)
    a_t = e_zi / np.sum(e_zi, axis=0, keepdims=True)
    return (a_t, z_t, a_p, z_p)
    

In [114]:
# Forward pass is working
players_1 = np.ones((1, 9))
players_1 = np.tile(players_1, (22, 1))
w_p = np.ones((1, 9))
b_p = 0
w_t = np.ones((121, 22))
b_t = 0
result_1 = f_x(
    players_1,
    w_p,
    b_p,
    w_t,
    b_t
)
print(result_1[0])

[[0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00

In [115]:
def L(y_pred, y):
    """
    Returns the means squared error of a single example
    
    Args:
        y_pred (ndarray) - predicted home away goal scores
        y (ndarray) - actual home away goal scores
        
    Returns:
        error (scalar) - mean squared error loss
    """
    loss = np.where(y == 1, -np.log(y_pred + 1e-15), 0) # Adding by a small number to prevent trying to get log of 0
    return np.sum(loss)

In [116]:
print(L(np.array([0.778,0.232]), np.array([1,0])))

0.25102875480374415


In [117]:
def calculate_gradients(
    X: npt.NDArray, 
    w_p: npt.NDArray, 
    b_p: float, 
    w_t: npt.NDArray, 
    b_t: float,
    Y: npt.NDArray
):
    """ 
    Function to calculate the gradients used in gradient descent
    
    Args:
        X (ndarray) - a (m,22,9) array with m training examples
        Y (ndarray) - a (m, 121, 1) array with labels for m training examples
        w_p (ndarray) - weights for the player section of neural network
        b_p (scalar) - bias for the player section of neural network
        w_t (ndarray) - weights for team section of neural network
        b_t (scalar) - bias for team section of neural network
        
    Returns:
        gradients (list) - list of gradients of parameters
    """
    dw_t = np.zeros(w_t.shape)
    db_t = 0
    dw_p = np.zeros(w_p.shape)
    db_p = 0
    J = 0
    
    m = X.shape[0]
    
    # For each training example
    for i in range(m):
        # Get the prediction
        x_i = X[i]
        y_i = Y[i]
        (y_pred_i, z_t_i, a_p_i, z_p_i) = f_x(
            x_i,
            w_p,
            b_p,
            w_t,
            b_t
        )
        
        
        # Get derivative
        # da_t_i = 2 * (y_pred_i - y_i)
        dz_t_i = y_pred_i - y_i
        dw_t += a_p_i.T * dz_t_i
        db_t += np.sum(dz_t_i)
        da_p_i = np.matmul(w_t.T, dz_t_i)
        dz_p_i = np.where(z_p_i < 0, 0, da_p_i)
        dw_p += np.matmul(dz_p_i.T, x_i)
        db_p += np.sum(dz_p_i)
        
        # Add loss 
        J += L(y_pred_i, y_i)
        
        
    dw_t /= m
    db_t /= m
    dw_p /= m
    db_p /= m
    J /= m
    return (dw_p, db_p, dw_t, db_t, J)

In [118]:
x_train_1 = np.ones((1, 9))
x_train_1 = np.tile(x_train_1, (22, 1))
w_p = np.ones((1, 9))
b_p = 0
w_t = np.ones((121, 22))
b_t = 0
X_train = np.array([
    x_train_1
])
Y_train = np.zeros((121, 1))
Y_train[0][0] = 1

print(calculate_gradients(
    X_train,
    w_p,
    b_p,
    w_t,
    b_t,
    Y_train
))

(array([[-2640., -2640., -2640., -2640., -2640., -2640., -2640., -2640.,
        -2640.]]), np.float64(-2639.9999999999986), array([[-8.92561983, -8.92561983, -8.92561983, ..., -8.92561983,
        -8.92561983, -8.92561983],
       [-8.92561983, -8.92561983, -8.92561983, ..., -8.92561983,
        -8.92561983, -8.92561983],
       [-8.92561983, -8.92561983, -8.92561983, ..., -8.92561983,
        -8.92561983, -8.92561983],
       ...,
       [-8.92561983, -8.92561983, -8.92561983, ..., -8.92561983,
        -8.92561983, -8.92561983],
       [-8.92561983, -8.92561983, -8.92561983, ..., -8.92561983,
        -8.92561983, -8.92561983],
       [-8.92561983, -8.92561983, -8.92561983, ..., -8.92561983,
        -8.92561983, -8.92561983]], shape=(121, 22)), np.float64(-120.00000000000001), np.float64(580.2906560171909))


In [119]:
# Splitting data into train / cross-validation / test sets
X = np.load("data/players_batch_6_2026-05-25 21:44:14.801126.npy")
Y = np.load("data/results_batch_6_2026-05-25 21:44:14.801174.npy")
X_train, X_mid, Y_train, Y_mid = train_test_split(X, Y, test_size=0.2, random_state=42)
X_cv, X_test, Y_cv, Y_test = train_test_split(X_mid, Y_mid, test_size=0.5, random_state=42)
print(X_train.shape, X_cv.shape, X_test.shape)
print(Y_train.shape, Y_cv.shape, Y_test.shape)

(560, 22, 9) (70, 22, 9) (70, 22, 9)
(560, 121, 1) (70, 121, 1) (70, 121, 1)


In [120]:
# Scale features
mean = np.mean(X_train, axis=(0,1), keepdims=True)
std = np.std(X_train, axis=(0,1), keepdims=True)
std = np.where(std == 0, 1.0, std)

X_train_scaled = (X_train - mean) / std
X_cv_scaled = (X_cv - mean) / std
X_test_scaled = (X_test - mean) / std
print(X_train_scaled.shape)
print(X_cv_scaled.shape)
print(X_test_scaled.shape)

(560, 22, 9)
(70, 22, 9)
(70, 22, 9)


In [121]:
def gradient_descent(
    X: npt.NDArray, 
    w_p: npt.NDArray, 
    b_p: float, 
    w_t: npt.NDArray, 
    b_t: float,
    Y: npt.NDArray,
    alpha: float,
    num_iters: int
):
    """ 
    This function will minimize the cost of the model using gradient descent and return found weights
    
    Args:
        X (ndarray) - a (m, 22, 9) array serving as input to model
        w_p (ndarray) - a (1,9) array with the initial weights of the player section of model
        b_p (scalar) - initial bias for player section of model
        w_t (ndarray) - a (2,22) array with initial weights of the team section of model
        b_t (scalar) - initial bias for team section of model
        Y (ndarray) - a (m, 2, 1) array with target outputs for model
        alpha (scalar) - learning rate
        num_iters (scalar) - number of times to run gradient descent
        
    Return:
        w_p (ndarray) - a (1, 9) array that minimizes cost
        b_p (scalar) - bias that minimized cost
        w_t (ndarray) - a (2, 22) array that minimizes cost
        b_t (scalar) - bias that minimizes cost
    """
    for i in range(num_iters):
        # Get derivatives
        (dw_p, db_p, dw_t, db_t, J) = calculate_gradients(
            X=X,
            w_p=w_p,
            b_p=b_p,
            w_t=w_t,
            b_t=b_t,
            Y=Y
        )
        
        # Update weights
        w_p = w_p - (alpha * dw_p)
        w_t = w_t - (alpha * dw_t)
        b_p = b_p - (alpha * db_p)
        b_t = b_t - (alpha * db_t)
        
        # Print result
        print(f"Cost at epoch {i}/{num_iters} = {J}")
        
    return (w_p, w_t, b_p, b_t)

In [147]:
# Get initial weights
w_t = np.random.rand(121, 22)
w_p = np.random.rand(1, 9)
b_p = 0
b_t = 0

# Run gradient descent
(new_w_p, new_w_t, new_b_p, new_bt) = gradient_descent(
    X=X_train_scaled,
    w_p=w_p,
    w_t=w_t,
    b_p=b_p,
    b_t=b_t,
    Y=Y_train,
    alpha=0.01,
    num_iters=100
)

Cost at epoch 0/100 = 8.499620678562902
Cost at epoch 1/100 = 8.281507294452913
Cost at epoch 2/100 = 8.07678595511298
Cost at epoch 3/100 = 7.884252133749179
Cost at epoch 4/100 = 7.702878948172411
Cost at epoch 5/100 = 7.531931592054317
Cost at epoch 6/100 = 7.3706400557799645
Cost at epoch 7/100 = 7.21840495219039
Cost at epoch 8/100 = 7.0746168365436635
Cost at epoch 9/100 = 6.93866029149269
Cost at epoch 10/100 = 6.810182753607327
Cost at epoch 11/100 = 6.6887569844776475
Cost at epoch 12/100 = 6.57393807195917
Cost at epoch 13/100 = 6.4651087175832185
Cost at epoch 14/100 = 6.3618006577545865
Cost at epoch 15/100 = 6.263696465244026
Cost at epoch 16/100 = 6.170589674782975
Cost at epoch 17/100 = 6.082186705113396
Cost at epoch 18/100 = 5.998205161644914
Cost at epoch 19/100 = 5.918393537274496
Cost at epoch 20/100 = 5.842521126254707
Cost at epoch 21/100 = 5.770400328778405
Cost at epoch 22/100 = 5.701791242765343
Cost at epoch 23/100 = 5.636416769917735
Cost at epoch 24/100 = 5.

In [148]:
def J(
    X: npt.NDArray, 
    w_p: npt.NDArray, 
    b_p: float, 
    w_t: npt.NDArray, 
    b_t: float,
    Y: npt.NDArray
):
    """ 
    Evaluate model on dataset
    
    Args:
        X (ndarray) - a (m,22,9) array with m examples
        Y (ndarray) - a (m, 2, 1) array with labels for m examples
        w_p (ndarray) - weights for the player section of neural network
        b_p (scalar) - bias for the player section of neural network
        w_t (ndarray) - weights for team section of neural network
        b_t (scalar) - bias for team section of neural network
    """
    J = 0
    m = X.shape[0]
    
    # For each training example
    for i in range(m):
        # Get the prediction
        x_i = X[i]
        y_i = Y[i]
        (y_pred_i, _, _, _) = f_x(
            x_i,
            w_p,
            b_p,
            w_t,
            b_t
        )
        
        # Add loss 
        J += L(y_pred_i, y_i)
        
    J /= m
    print(f"Cost is {J}")
    

In [149]:
J(
    X=X_cv,
    w_p=new_w_p,
    b_p=new_b_p,
    w_t=new_w_t,
    b_t=new_bt,
    Y=Y_cv
)

Cost is 29.586521120050747


In [150]:
def interpret_probabilities(prob):
    """ 
    Function to interpret the returned probabilities by the model
    
    Args:
        prob (ndarray) - array with probabilities for different match results
        
    Returns:
        scores (str) - human readable version of prediction
    """
    predicted = np.argmax(prob)
    home_score = math.floor(predicted / 11)
    away_score = predicted % 11
    return f"{home_score}-{away_score}"

In [151]:
prob = f_x(
    X_cv_scaled[5],
    new_w_p,
    new_b_p,
    new_w_t,
    new_bt
)[0]

predicted_score = interpret_probabilities(prob)
actual_score = interpret_probabilities(Y_cv[5])
print(f"Predicted score is {predicted_score} while actual is {actual_score}")

Predicted score is 2-0 while actual is 5-0


In [152]:
def accuracy_score(y_pred, y_label):
    """ 
    Return the ratio of correct responses across the entire test set.add
    
    Args:
        y_pred (ndarray): (m, 121, 1) array with predictions for each class from the model from the model
        y_label(ndarray): (m, 121, 1) array with the actual labels
        
    Returns:
        accuracy (scalar): ratio of correct responses to all responses
    """
    predictions = np.argmax(y_pred, axis=1)
    actual = np.argmax(y_label, axis=1)
    num_correct = np.sum(predictions == actual)
    return num_correct / y_pred.shape[0]

In [153]:
def predict(
    X: npt.NDArray, 
    w_p: npt.NDArray, 
    b_p: float, 
    w_t: npt.NDArray, 
    b_t: float,
):
    """ 
    Make prediction for all of the examples in the passed dataset
    
    Args:
        X (ndarray) - a (m,22,9) array with m examples
        w_p (ndarray) - weights for the player section of neural network
        b_p (scalar) - bias for the player section of neural network
        w_t (ndarray) - weights for team section of neural network
        b_t (scalar) - bias for team section of neural network
        
    Returns:
        Y_pred (ndarray) - a (m, 121, 1) array with prediction of all examples in X
    """
    m = X.shape[0]
    predictions = []
    
    for i in range(m):
        x_i = X[i]
        (pred, _, _, _) = f_x(
            x_i,
            w_p,
            b_p,
            w_t,
            b_t
        )
        predictions.append(pred)
        
    Y_pred = np.array(predictions)
    return Y_pred

In [154]:
y_pred = predict(
    X_cv_scaled,
    new_w_p,
    new_b_p,
    new_w_t,
    new_bt
)
accuracy = accuracy_score(y_pred, Y_cv)
print(f"Accuracy of model is {accuracy}")

Accuracy of model is 0.14285714285714285


In [155]:
class PredictionRecall:
    def __init__(self, t_p: int, f_p: int, f_n:int, total: int, label: str):
        """ 
        Return object with prediction recall values
        
        Args:
            label (str): Name of the class
            t_p (int): Total true positive for the given label
            f_p (int): Total false positives for given label
            total (int): Total items with given label
            f_n (int): Total false negatives for given label
        """
        self.total = total
        self.t_p = t_p
        self.f_p = f_p
        self.f_n = f_n
        self.label = label
        
    def __repr__(self):
        return f"{self.label} with t_p: {self.t_p}, f_p: {self.f_p}, f_n: {self.f_n} and total: {self.total}"

def get_precision_recall(y_pred, y_label) -> list[PredictionRecall]:
    """ 
    Returns the precision and recall of the model based on the predictions it has made
    
    Args:
        y_pred (ndarray) - a (m, 121, 1) array with the predictions of the model
        y_label (ndarray) - a (m, 121, 1) array with the actual values
        
    Returns:
        prediction_recall (list) - list of prediction recall values
    """
    predictions = [interpret_probabilities(prob) for prob in y_pred]
    actual = [interpret_probabilities(prob) for prob in y_label]
    all_items = predictions.copy()
    all_items.extend(actual)
    labels = set(all_items)
    
    m = len(actual)
    precisions = []
    
    for label in labels:
        total = 0
        false_positive = 0
        false_negative = 0
        true_positive = 0
        
        for i in range(m):
            entry = actual[i]
            pred = predictions[i]
            
            if entry == label:
                total += 1
                
            if entry == label and pred == entry:
                true_positive += 1
            elif entry != label and pred == label:
                false_positive += 1
        false_negative = total - true_positive
        
        precisions.append(PredictionRecall(
            t_p=true_positive,
            f_p=false_positive,
            f_n=false_negative,
            total=total,
            label=label
        ))
    
    return precisions
    

In [156]:
precisions = get_precision_recall(
    y_pred=np.array([
        [
            [0],
            [0],
            [0]
        ],
        [
            [0],
            [0],
            [0]
        ],
        [
            [0],
            [0],
            [0]
        ],
        [
            [0],
            [0],
            [0]
        ],
        [
            [0],
            [0],
            [0]
        ],
        [
            [0],
            [0],
            [0]
        ],
        [
            [0],
            [0],
            [0]
        ],
        [
            [0],
            [0],
            [0]
        ],
        [
            [0],
            [0],
            [0]
        ]]),
    y_label=np.array([
        [
            [0],
            [0],
            [0]
        ],
        [
            [0],
            [0],
            [0]
        ],
        [
            [0],
            [1],
            [0]
        ],
        [
            [0],
            [0],
            [2]
        ],
        [
            [0],
            [1],
            [0]
        ],
        [
            [0],
            [1],
            [0]
        ],
        [
            [0],
            [0],
            [2]
        ],
        [
            [0],
            [0],
            [0]
        ],
        [
            [0],
            [1],
            [0]
        ]]),
)
print(precisions)

[0-2 with t_p: 0, f_p: 0, f_n: 2 and total: 2, 0-1 with t_p: 0, f_p: 0, f_n: 4 and total: 4, 0-0 with t_p: 3, f_p: 6, f_n: 0 and total: 3]


In [161]:
def precision_recall(y_pred, y_label):
    """ 
        Print precision recall for model predictions
        
        Args:
            y_pred (ndarray): a (m, 121, 1) array with the predictions of the model
            y_label (ndarray): a (m, 121, 1) array with labels of the model
    """ 
    precisions = get_precision_recall(y_pred=y_pred, y_label=y_label)
    
    # Order precisions by totals
    n = len(precisions)
    for i in range(n):
        swapped = False
        
        for j in range(0, n-i-1):
            # Traverse array from 0 to n - i - 1
            # Swap if the element found is lesser than next element
            if precisions[j].total < precisions[j+1].total:
                precisions[j], precisions[j + 1] = precisions[j + 1], precisions[j]
                swapped = True
        
        if (swapped == False):
            break
    
    
    print(f"    Precision and recall")
    for prec in precisions:
        precision = 0 if prec.f_p + prec.t_p == 0 else (prec.t_p / (prec.f_p + prec.t_p))
        recall = 0 if prec.t_p + prec.f_n == 0 else (prec.t_p) / (prec.t_p + prec.f_n)
        p_1 = 0 if precision == 0 else 1 / precision
        r_1 = 0 if recall == 0 else 1 / recall
        f1_score = 1 / 0.5 * (p_1 + r_1)
        print(f"{prec.label.center(5)}  TP:{prec.t_p}  FP:{prec.f_p}  FN:{prec.f_n}: Prec:{precision:.4f}  Rec:{recall:.4f}  F1:{f1_score:.4f}  Total:{prec.total}")

In [162]:
precision_recall(y_pred=y_pred, y_label=Y_cv)

    Precision and recall
 2-0   TP:2  FP:4  FN:18: Prec:0.3333  Rec:0.1000  F1:26.0000  Total:20
 1-0   TP:0  FP:0  FN:14: Prec:0.0000  Rec:0.0000  F1:0.0000  Total:14
 4-0   TP:4  FP:20  FN:9: Prec:0.1667  Rec:0.3077  F1:18.5000  Total:13
 3-0   TP:4  FP:24  FN:7: Prec:0.1429  Rec:0.3636  F1:19.5000  Total:11
 5-0   TP:0  FP:0  FN:6: Prec:0.0000  Rec:0.0000  F1:0.0000  Total:6
 0-0   TP:0  FP:2  FN:5: Prec:0.0000  Rec:0.0000  F1:0.0000  Total:5
 6-0   TP:0  FP:0  FN:1: Prec:0.0000  Rec:0.0000  F1:0.0000  Total:1
 2-4   TP:0  FP:2  FN:0: Prec:0.0000  Rec:0.0000  F1:0.0000  Total:0
 9-7   TP:0  FP:1  FN:0: Prec:0.0000  Rec:0.0000  F1:0.0000  Total:0
 8-2   TP:0  FP:1  FN:0: Prec:0.0000  Rec:0.0000  F1:0.0000  Total:0
 1-6   TP:0  FP:1  FN:0: Prec:0.0000  Rec:0.0000  F1:0.0000  Total:0
 8-3   TP:0  FP:2  FN:0: Prec:0.0000  Rec:0.0000  F1:0.0000  Total:0
 6-1   TP:0  FP:2  FN:0: Prec:0.0000  Rec:0.0000  F1:0.0000  Total:0
 1-7   TP:0  FP:1  FN:0: Prec:0.0000  Rec:0.0000  F1:0.0000  Total:

In [163]:
# Debugging f_x
print(w_p)

[[0.90392568 0.95142879 0.10466867 0.33179473 0.95904752 0.75161354
  0.76862601 0.82776568 0.11400258]]


In [164]:
print(new_w_p)
print(new_w_p.shape)

[[0.40456307 0.68001351 0.06160601 0.39795114 1.1140935  0.21094899
  0.1760306  0.22628309 0.07350406]]
(1, 9)


In [165]:
print(new_b_p)

-0.29635112214046994


In [166]:
print(w_t)

[[0.54944375 0.68890135 0.18095413 ... 0.98821604 0.8000955  0.34011265]
 [0.95856233 0.29084847 0.21758606 ... 0.38012227 0.80604989 0.96010909]
 [0.64966078 0.74473258 0.15516312 ... 0.54380332 0.93703509 0.4341881 ]
 ...
 [0.24293836 0.78474358 0.72539022 ... 0.27293611 0.28678901 0.02678094]
 [0.81782746 0.62125098 0.75850078 ... 0.43898933 0.7976675  0.01788683]
 [0.0428146  0.29702821 0.91175868 ... 0.78425524 0.02833735 0.47403984]]


In [167]:
print(new_w_t)

[[0.7029471  0.70370267 0.19116063 ... 1.03192928 0.8505456  0.37355002]
 [0.94357438 0.29016115 0.21690306 ... 0.37650172 0.80239182 0.95523439]
 [0.63426116 0.74355548 0.15448447 ... 0.53919459 0.93212229 0.4319839 ]
 ...
 [0.23274188 0.78214323 0.72403092 ... 0.26973931 0.28477315 0.02550682]
 [0.79015174 0.61917806 0.75649056 ... 0.43163346 0.79144707 0.01557747]
 [0.04008395 0.29673448 0.91117648 ... 0.78229807 0.02794601 0.47327276]]


In [139]:
print(new_bt)

8.756497133678728e-19


In [168]:
example = X_cv_scaled[5]
print(example)

[[ 1.99280815e-01 -5.31788336e-01 -7.16858058e-01 -8.06865475e-01
   3.12557687e+00 -6.30667579e-01 -3.47737832e-02 -7.20536566e-01
  -8.80149937e-03]
 [ 1.99280815e-01 -5.31788336e-01 -7.16858058e-01  1.23936397e+00
  -3.19940939e-01 -6.30667579e-01  4.41017644e-01 -5.64417802e-01
   2.75347935e-02]
 [-3.45458399e-01 -5.31788336e-01 -7.16858058e-01  1.23936397e+00
  -3.19940939e-01 -6.30667579e-01 -1.46287399e-01 -2.91209965e-01
  -4.91751582e-02]
 [-3.45458399e-01 -5.31788336e-01  1.39497630e+00 -8.06865475e-01
  -3.19940939e-01 -6.30667579e-01  3.21343863e-02  4.11324473e-01
   1.12319477e-01]
 [ 1.28875924e+00 -5.31788336e-01 -7.16858058e-01  1.23936397e+00
  -3.19940939e-01 -6.30667579e-01  3.22069787e-01 -2.71695120e-01
   7.34796414e-03]
 [ 1.28875924e+00 -5.31788336e-01 -7.16858058e-01  1.23936397e+00
  -3.19940939e-01 -6.30667579e-01  5.22794296e-01 -5.25388111e-01
  -7.26767615e-04]
 [-6.17828006e-01 -5.31788336e-01  1.39497630e+00 -8.06865475e-01
  -3.19940939e-01 -6.3066757

In [169]:
pred, z_t, a_p, z_p = f_x(
    example,
    new_w_p,
    new_b_p,
    new_w_t,
    new_bt
)

In [171]:
print(pred)
print(interpret_probabilities(pred))
print(interpret_probabilities(Y_cv[5]))

[[0.01793477]
 [0.00951411]
 [0.01571443]
 [0.00479168]
 [0.00487157]
 [0.00409368]
 [0.00758394]
 [0.00249994]
 [0.00852612]
 [0.00095716]
 [0.00874717]
 [0.01826656]
 [0.00207255]
 [0.01738334]
 [0.00117368]
 [0.00443661]
 [0.00456365]
 [0.0062947 ]
 [0.01863561]
 [0.00138759]
 [0.00814502]
 [0.00352784]
 [0.05830822]
 [0.00263429]
 [0.00264945]
 [0.0031624 ]
 [0.00632305]
 [0.00777177]
 [0.005499  ]
 [0.00608477]
 [0.01654469]
 [0.00426008]
 [0.00322771]
 [0.0281557 ]
 [0.00341948]
 [0.0035744 ]
 [0.00784384]
 [0.00115048]
 [0.00187485]
 [0.00293747]
 [0.01068206]
 [0.00064906]
 [0.00248519]
 [0.00755383]
 [0.05453873]
 [0.00339512]
 [0.00868143]
 [0.0039204 ]
 [0.00659633]
 [0.01013615]
 [0.0033574 ]
 [0.00782647]
 [0.00300848]
 [0.00251655]
 [0.02387044]
 [0.00178652]
 [0.00534958]
 [0.0010917 ]
 [0.00388607]
 [0.00395385]
 [0.01583195]
 [0.00246465]
 [0.00369766]
 [0.00287194]
 [0.0050424 ]
 [0.00787957]
 [0.02003093]
 [0.03082307]
 [0.00641359]
 [0.0072966 ]
 [0.00679092]
 [0.01

In [172]:
print(z_t)

[[3.74878483]
 [3.11481893]
 [3.6166228 ]
 [2.42892434]
 [2.44545887]
 [2.27148726]
 [2.88807684]
 [1.77830999]
 [3.00517778]
 [0.81825537]
 [3.03077336]
 [3.76711528]
 [1.59082534]
 [3.71755552]
 [1.02218848]
 [2.35193488]
 [2.38016524]
 [2.70175101]
 [3.78711754]
 [1.18960991]
 [2.9594495 ]
 [2.12272976]
 [4.92778648]
 [1.83065677]
 [1.83639616]
 [2.01337454]
 [2.70624542]
 [2.91254182]
 [2.56660886]
 [2.66783249]
 [3.66810858]
 [2.31133018]
 [2.03381567]
 [4.19979319]
 [2.09153317]
 [2.13583983]
 [2.92177204]
 [1.00222429]
 [1.49057402]
 [1.93959351]
 [3.23060938]
 [0.42982086]
 [1.7723916 ]
 [2.88409762]
 [4.86095457]
 [2.08438267]
 [3.02322995]
 [2.22823686]
 [2.74855732]
 [3.17815126]
 [2.07321151]
 [2.91955481]
 [1.96347793]
 [1.78493225]
 [4.03468416]
 [1.44231144]
 [2.53906145]
 [0.94977947]
 [2.21944099]
 [2.23673193]
 [3.62407317]
 [1.76409316]
 [2.16974359]
 [1.91703044]
 [2.47992555]
 [2.9263163 ]
 [3.85932088]
 [4.29030669]
 [2.72046318]
 [2.84945256]
 [2.77762959]
 [3.35

In [173]:
print(a_p)

[[2.23672424]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [2.20264067]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.84015852]]


In [174]:
print(z_p)

[[ 2.23672424]
 [-0.66585367]
 [-0.93343488]
 [-1.41538266]
 [-0.18127502]
 [-0.20394132]
 [-1.66528577]
 [-0.93850742]
 [-1.55636424]
 [-0.0847857 ]
 [-1.78626182]
 [ 2.20264067]
 [-1.3167236 ]
 [-0.73553635]
 [-2.06209794]
 [-0.84287811]
 [-1.40798219]
 [-0.46995527]
 [-0.76947337]
 [-0.63185331]
 [-1.98173455]
 [ 0.84015852]]


In [146]:
print(1 / 121)

0.008264462809917356
